# Spark ML — renewal classifier (PySpark `Pipeline`)

**Production alternative** to `python/cs_portfolio/train.py` + sklearn: same **silver** table (`lease_episode_features`), same **numeric + `metro`** features, label **`renewed`**. Trains **`pyspark.ml.classification.LogisticRegression`** inside a **`Pipeline`** (StringIndexer → OneHotEncoder → VectorAssembler).

**Prereqs:** run `01_feature_pipeline` (or local pipeline) so Parquet exists. Set `PORTFOLIO_REPO_ROOT` / `PORTFOLIO_BASE_PATH` like other notebooks in this repo.

In [ ]:
import os
from pathlib import Path

from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

REPO = Path(os.environ.get("PORTFOLIO_REPO_ROOT", "../..")).resolve()
base = os.environ.get("PORTFOLIO_BASE_PATH")
if base:
    silver = f"{base.rstrip('/')}/silver/lease_episode_features.parquet"
else:
    silver = str(REPO / "data/silver/lease_episode_features.parquet")

print("Silver path:", silver)

raw = spark.read.parquet(silver)

num_cols = [
    "tenure_months",
    "monthly_rent",
    "rent_to_market_ratio",
    "late_payment_count_12m",
    "work_order_count_12m",
    "prior_renewal_count",
    "bedrooms",
    "is_long_tenure",
    "high_service_load",
]

for c in num_cols:
    raw = raw.withColumn(c, F.col(c).cast("double"))

grain = ["lease_id"] if "lease_id" in raw.columns else []
df = raw.select(*(grain + num_cols + ["metro", "renewed"])).dropna(subset=["renewed"])
df = df.withColumn("renewed", F.col("renewed").cast("double"))

train_df, test_df = df.randomSplit([0.75, 0.25], seed=42)
print("train", train_df.count(), "test", test_df.count())

In [ ]:
metro_indexer = StringIndexer(
    inputCol="metro", outputCol="metro_idx", handleInvalid="keep"
)
metro_ohe = OneHotEncoder(
    inputCols=["metro_idx"], outputCols=["metro_vec"], dropLast=True
)
assembler = VectorAssembler(
    inputCols=num_cols + ["metro_vec"],
    outputCol="features",
    handleInvalid="skip",
)
lr = LogisticRegression(
    featuresCol="features",
    labelCol="renewed",
    maxIter=200,
    family="binomial",
    standardization=True,
)

pipeline = Pipeline(stages=[metro_indexer, metro_ohe, assembler, lr])
model = pipeline.fit(train_df)

pred = model.transform(test_df)
evaluator = BinaryClassificationEvaluator(
    rawPredictionCol="rawPrediction",
    labelCol="renewed",
    metricName="areaUnderROC",
)
auc = evaluator.evaluate(pred)
print(f"Test AUROC: {auc:.4f}")

show_cols = [c for c in ("lease_id",) if c in pred.columns] + ["renewed", "probability", "prediction"]
pred.select(*show_cols).show(5, truncate=False)

## Optional — MLflow (`pyspark.ml`)

Uncomment when your workspace experiment is configured. Registers a **Spark ML** flavor for UC registry later (`databricks/docs/PRODUCTIONIZATION_UC_MLFLOW.md`).

In [ ]:
# import mlflow
# from mlflow.spark import log_model
#
# mlflow.set_experiment("/Shared/portfolio_renewal_demo_sparkml")
# with mlflow.start_run(run_name="sparkml_logistic"):
#     mlflow.log_param("silver_path", silver)
#     mlflow.log_metric("test_auroc", float(auc))
#     log_model(model, artifact_path="sparkml-pipeline")
# print("Done — register in UC when catalog + permissions are ready.")